# EVA — Yandex Cloud Training
## 32 GB VRAM, 128-dim, 32 heads, 6 layers, ~1.2M params

**Инструкция:**
1. Загрузить репозиторий на VM
2. Загрузить `real_data/connected_ru.npy` (503M токенов)
3. Загрузить чекпоинты из `checkpoints/symbolic/`
4. Запустить все ячейки по порядку

In [ ]:
# Cell 1: Setup
!pip install torch numpy scikit-learn loguru psutil conceptnet-lite -q

import torch, sys, os
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)")

In [ ]:
# Cell 2: Verify data
import numpy as np
data = np.load('real_data/connected_ru.npy', mmap_mode='r').astype(np.int32)
print(f"Corpus: {len(data)/1e6:.1f}M tokens")

import os
ckpt_dir = 'checkpoints/symbolic'
for f in sorted(os.listdir(ckpt_dir)):
    size = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Cell 3: Run training
!python train_yandex.py

In [ ]:
# Cell 4: Monitor progress
with open('yandex_train_log.txt', 'r') as f:
    lines = f.readlines()
    for line in lines[-20:]:
        print(line.rstrip())

In [ ]:
# Cell 5: Test generation
import torch, torch.nn.functional as F
from eva.symbolic.char_vocab import CharacterVocab
from eva.symbolic.unified_transformer import UnifiedMultidimensionalTransformer

cv = CharacterVocab(); VT = 157; DEVICE = 'cuda'
ut = UnifiedMultidimensionalTransformer(vocab_size=157, coord_dim=128,
    num_levels=8, scales_per_level=4, num_layers=6, d_ff=512).to(DEVICE)
ckpt = torch.load('checkpoints/symbolic/yandex_latest.pt', map_location='cpu')
ut.load_state_dict(ckpt['ut'], strict=False)
ut.eval()

def gen(ids, n=30, T=0.8):
    ids = list(ids)
    with torch.no_grad():
        for _ in range(n):
            _, sc = ut(torch.tensor([ids], dtype=torch.long, device=DEVICE), return_scores=True)
            logits = sc[0, -1] / T
            sl, si = logits.sort(descending=True)
            cp = F.softmax(sl, dim=-1).cumsum(dim=-1)
            cut = (cp > 0.95).nonzero(as_tuple=True)[0]
            k = cut[0].item() + 1 if len(cut) > 0 else 30
            k = min(max(k, 3), 50)
            v, idx = logits.topk(k); p = F.softmax(v, dim=-1)
            for t in set(ids[-5:]):
                m = (idx == t).nonzero(as_tuple=True)[0]
                if len(m) > 0: p[m] *= 0.2
            p /= p.sum()
            nt = idx[torch.multinomial(p, 1)].item()
            if nt <= 0 or nt >= VT: nt = idx[0].item()
            ids.append(nt)
    return ids

for w in ['привет', 'человек идет', 'солнце светит', 'сегодня хорошая', 'я люблю']:
    ids = cv.encode(w)[1:-1]
    if len(ids) >= 2:
        r = gen(ids, 30, 0.8)
        print(f"{w} -> {cv.decode(r)}")